## 라이브러리

In [ ]:
from __future__ import annotations
from price_windows import event_price_window, market_timestamp
from typing import List, Dict, Any, Tuple, Annotated, Optional
from typing_extensions import TypedDict
from dataclasses import dataclass
from datetime import datetime, timezone, timedelta
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
import operator
import math
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import csv, json, os
from tabulate import tabulate
from langchain_core.documents import Document
import pandas as pd
import re
from rank_bm25 import BM25Okapi
import requests, html, time, pytz
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams
from FlagEmbedding import BGEM3FlagModel
import yfinance as yf

### 환경설정

In [ ]:
load_dotenv()

llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

bge_m3 = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True,   # GPU 있으면 True, CPU만 쓰면 False 로 바꿔도 됨
)

NAVER_NEWS_ENDPOINT = "https://openapi.naver.com/v1/search/news.json"
_TAG_RE = re.compile(r"<.*?>")

df = pd.read_csv("data\\naver_economy_news_clean.csv")

price_df = pd.read_csv("data\\price_long_3y.csv", parse_dates=["date_kst"])
price_df["date_kst"] = pd.to_datetime(price_df["date_kst"], utc=True).dt.tz_localize(None)
price_df["ticker"] = price_df["ticker"].astype(str).str.upper()

ETF_LIST = ["ARKK", "BITO", "DIA", "GLD", "QQQ", "SLV", "SPY", "TQQQ", "XLE", "XLF"]
ALLOWED_ETFS = {"ARKK","BITO","DIA","GLD","QQQ","SLV","SPY","TQQQ","XLE","XLF"}

KNOWN_TICKERS = set(price_df["ticker"].unique().tolist())

KEYWORD_TO_TICKERS = {
    # 지수/지수계열
    r"\b나스닥\b|\bNasdaq\b|QQQ|NDX": ["QQQ", "TQQQ"],
    r"\bS&P\b|에스앤피|에스피|SP500|S&P500|SP 500|SPX": ["SPY"],
    r"\b다우\b|Dow|다우존스|DJI": ["DIA"],
    # 섹터
    r"\b에너지\b|정유|원유|셰일|Oil|WTI|브렌트": ["XLE"],
    r"\b금융\b|은행|보험|증권|Financials|Banks": ["XLF"],
    # 테마/자산군
    r"\b금\b|Gold|골드|금값|금리와 금|금 ETF": ["GLD"],
    r"\b은\b|Silver|실버": ["SLV"],
    r"\b비트코인\b|Bitcoin|BTC|코인|가상자산": ["BITO"],
    r"\b혁신\b|Cathie\s*Wood|아크|ARK|고성장|디스럽티브": ["ARKK"],
    # 레버리지
    r"\bTQQQ\b|3배 레버리지|레버리지 나스닥": ["TQQQ"],
}

_CLEAN_RE = re.compile(r"\s+")

HIST_TOPK = 3
HIST_MIN_DAYS_GAP = 30

## 한글 폰트

In [ ]:
def use_korean_font():
    candidates = ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]
    # 1) 이미 설치된 폰트에서 찾기
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            rcParams["font.family"] = name
            rcParams["axes.unicode_minus"] = False  # 마이너스 깨짐 방지
            return name

    # 2) (선택) 폰트 파일 경로가 있다면 등록해서 사용
    # font_manager.fontManager.addfont("C:/Windows/Fonts/malgun.ttf")
    # rcParams["font.family"] = "Malgun Gothic"
    rcParams["axes.unicode_minus"] = False
    return None

In [ ]:
chosen = use_korean_font()
print("Korean font set:", chosen)

In [ ]:
try:
    LOCAL_DOCS  # noqa
except NameError:
    df = pd.read_csv("data/naver_economy_news_clean.csv")

    def _row_to_doc(r: pd.Series) -> Document:
        # NaN -> "" 처리 + 문자열 변환
        raw_title = r.get("title", "")
        raw_desc  = r.get("description", "")
        raw_date  = r.get("published_at_utc", r.get("date_kst", ""))

        title = "" if pd.isna(raw_title) else str(raw_title)
        desc  = "" if pd.isna(raw_desc)  else str(raw_desc)
        date  = "" if pd.isna(raw_date)  else str(raw_date)

        meta = {
            "date": date,
            "source": "local",
            "url": r.get("naver_link") or r.get("original_link"),
            "title": title,
        }

        txt = f"{title}\n\n{desc}".strip()
        return Document(page_content=txt, metadata=meta)

    LOCAL_DOCS = [_row_to_doc(r) for _, r in df.iterrows()]

In [ ]:
print("문서 개수:", len(LOCAL_DOCS))
print("첫 문서 내용 일부:\n", LOCAL_DOCS[0].page_content[:200])
print("첫 문서 메타데이터:\n", LOCAL_DOCS[0].metadata)

## 토크나이저

In [ ]:
_token_pat = re.compile(r"[0-9A-Za-z가-힣]+")

def tokenize_ko(text: str) -> List[str]:
    return _token_pat.findall(text.lower())

## BM25

In [ ]:
try:
    _BM25  # noqa
except NameError:
    _bm25_corpus_tokens = [tokenize_ko(d.page_content) for d in LOCAL_DOCS]
    _BM25 = BM25Okapi(_bm25_corpus_tokens)

## 데이터 파싱

In [ ]:
df = pd.read_csv("data\\naver_economy_news_clean.csv")

price_df = pd.read_csv("data\\price_long_3y.csv", parse_dates=["date_kst"])
price_df["date_kst"] = pd.to_datetime(price_df["date_kst"], utc=True).dt.tz_localize(None)
price_df["ticker"] = price_df["ticker"].astype(str).str.upper()

etf_profile_df = pd.read_csv("data/etf_profile.csv")
etf_profile_table = {
    row["ticker"]: row.to_dict()
    for _, row in etf_profile_df.iterrows()
}

docs = []
for _, row in df.iterrows():
    # 안전하게 문자열 처리 (NaN 방지)
    title = str(row.get("title", "")).strip()
    desc = str(row.get("description", "")).strip()

    # title이나 description이 NaN이면 빈 문자열로 처리
    if title == "nan":
        title = ""
    if desc == "nan":
        desc = ""

    text = f"{title}\n\n{desc}"

    metadata = {
        "date": str(row.get("published_at_utc", "")),  # 날짜
        "source": "Naver",                            # 출처 명시
        "url": row.get("naver_link") or row.get("original_link"),
        "query_used": row.get("query_used"),          # 수집 키워드
    }

    docs.append(Document(page_content=text, metadata=metadata))

print(f"✅ 문서 개수: {len(docs)}")
print(f"📄 예시 문서 내용:\n{docs[0].page_content[:200]}")
print(f"📑 메타데이터 예시:\n{docs[0].metadata}")
print('------------------------------------------------------------')
print("✅ 데이터 예시:")
print(price_df.head())

In [ ]:

def get_price_window(ticker: str, ref_date, window=3, *, as_of=None):
    """기사 시점 이전 마지막 종가 → 기사 이후 window개 관측 종가. window는 달력일이 아니다."""
    cutoff = as_of if as_of is not None else datetime.now(timezone.utc).isoformat()
    return event_price_window(price_df, ticker, ref_date, as_of=cutoff,
                              horizon_sessions=window)


## 스플리터

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", "!", "?", " "],
)
split_docs = splitter.split_documents(docs)
print(f"✅ 분할 후 청크 수: {len(split_docs)}")

## 임베딩

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 벡터스토어 생성

In [ ]:
os.makedirs("vectorstore", exist_ok=True)
vectorstore_path = "vectorstore/etf_news_faiss"

vectorstore = FAISS.from_documents(split_docs, embedding=embeddings)
vectorstore.save_local(vectorstore_path)

print(f"✅ FAISS 벡터스토어 저장 완료: {vectorstore_path}")

벡터 스토어 테스트

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.load_local("vectorstore/etf_news_faiss", embeddings, allow_dangerous_deserialization=True)

# 테스트
query = "금리 인하 전망에 따른 ETF 반응"
results = vectorstore.similarity_search(query, k=3)
for r in results:
    print(f"📄 {r.metadata['date']} | {r.page_content[:100]}...")

벡터 스토어와 그래프 연결

In [ ]:
# 임베딩은 벡터스토어 생성 시와 동일 모델 사용
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 저장해둔 경로 (예: vectorstore/etf_news_faiss)
VSTORE_PATH = "vectorstore/etf_news_faiss"

vectorstore = FAISS.load_local(
    VSTORE_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print("✅ VectorStore loaded:", VSTORE_PATH)

BM25 확인용

In [ ]:
# 디버깅용 간단 BM25 테스트
test_query = "금리 인하"  # 또는 "연준", "ETF 하락" 같이 네가 궁금한 키워드

tokens = tokenize_ko(test_query)
scores = _BM25.get_scores(tokens)

# 상위 5개 문서 인덱스와 점수
top = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:5]

print("=== BM25 상위 5개 결과 ===")
for rank, (idx, score) in enumerate(top, start=1):
    doc = LOCAL_DOCS[idx]
    print(f"\n[{rank}] score={score:.4f}")
    print("제목:", doc.metadata.get("title"))
    print("날짜:", doc.metadata.get("date"))
    print("본문 일부:", doc.page_content[:150].replace("\n", " "))

벡터스토어 그래프 연결 확인용

In [ ]:
test_query = "금리 인하"  # 같은 쿼리

dense_hits = vectorstore.similarity_search(test_query, k=5)

print("=== FAISS dense 상위 5개 ===")
for i, doc in enumerate(dense_hits, start=1):
    print(f"\n[{i}]")
    print("제목:", doc.metadata.get("title"))
    print("날짜:", doc.metadata.get("date"))
    print("본문 일부:", doc.page_content[:150].replace("\n", " "))


## 컨피그

In [ ]:
class RAGConfig:
    # === 기본 RAG 세팅 ===
    embedding_backend: str = "hf"       # "hf" or "openai"
    reranker_backend: str  = "hf"       # "hf" or "openai"
    use_naver: bool = True
    ensemble_topN: int = 80             # candidate pool size before rerank
    rerank_topk: int = 12               # final k
    rrf_k: int = 75                     # RRF smoothing
    thread_id: str = "ensemble-demo"

    # === 네이버 뉴스 API 설정 ===
    naver_client_id: str = os.getenv("NAVER_CLIENT_ID")        #  실제 발급받은 Client ID
    naver_client_secret: str = os.getenv("NAVER_CLIENT_SECRET") #  실제 발급받은 Client Secret
    naver_timeout: int = 10
    naver_page_size: int = 20
    naver_max_pages: int = 2

CFG = RAGConfig()

컨피그 점검용

In [ ]:
print("NAVER_CLIENT_ID:", "SET" if CFG.naver_client_id else "NOT SET")
print("NAVER_CLIENT_SECRET:", "SET" if CFG.naver_client_secret else "NOT SET")
print("use_naver:", CFG.use_naver)
print("ensemble_topN, rerank_topk, rrf_k:", CFG.ensemble_topN, CFG.rerank_topk, CFG.rrf_k)

## 스테이트 스키마

In [ ]:
class GraphState(TypedDict, total=False):
    question: str
    today: str
    as_of: str
    horizon_sessions: int
    rewritten_question: str
    bm25_hits: List[Dict[str, Any]]
    dense_hits: List[Dict[str, Any]]
    web_hits: List[Dict[str, Any]]
    fused_hits: List[Dict[str, Any]]
    reranked_hits: List[Dict[str, Any]]
    matched_etfs: List[str]
    price_context: str
    etf_stats: Dict[str, Any]
    price_window_status: Dict[str, Any]
    price_reactions: List[Dict[str, Any]]
    etf_profile: Dict[str, Any]
    macro_context: Dict[str, Any]
    news_impact_analysis: str
    historical_block: str
    consistency_block: str
    consistency_scores: Dict[str, float]
    consistency_score: Optional[float]
    consistency_comment: str
    final_answer: str


## 신규 노드 및 유틸

가격 반응 분석 노드

In [ ]:
def analyze_price_reaction(state: dict) -> dict:
    """이미 계산·검증한 동일 가격창 결과를 사용한다. 별도 close/price_df 입력은 사용하지 않는다."""
    reactions = []
    for ticker, stats in state.get("etf_stats", {}).items():
        change = stats.get("current_change")
        window = stats.get("current_window")
        if change is None or not window:
            continue
        change = float(change)
        direction = "UP" if change > 1.0 else "DOWN" if change < -1.0 else "FLAT" if abs(change) < 0.3 else "MIXED"
        reactions.append({"ticker": ticker, "start_date": window[0], "end_date": window[1],
                          "return_pct": change, "direction": direction})
    return {"price_reactions": reactions}


뉴스가 가격에 준 영향 분석 노드용

In [ ]:
def explain_news_impact(state: dict) -> dict:
    """
    가격 반응(PriceReaction), 상위 뉴스(reranked_hits), ETF 프로필(etf_profile)을 종합해
    '이번 가격 움직임에 뉴스가 어떤 영향을 줬는지'를 설명하는 중간 분석을 생성한다.
    결과는 state["news_impact_analysis"]에 저장된다.
    """

    reactions = state.get("price_reactions", []) or []
    if not reactions:
        return {"news_impact_analysis": "완료된 기사 이후 가격창이 없어 영향 설명을 생성하지 않았습니다."}
    hits = state.get("reranked_hits", []) or []
    question = state.get("question", "")
    etf_profile = state.get("etf_profile", {}) or {}
    macro_context = state.get("macro_context", None)  # 나중에 붙일 거 고려해서

    # --- ETF 프로필 텍스트 구성 ---
    etf_profile_text = ""
    if etf_profile:
        etf_profile_text = (
            f"티커: {etf_profile.get('ticker')}, 이름: {etf_profile.get('name')}\n"
            f"그룹: {etf_profile.get('group')} / 자산군: {etf_profile.get('asset_class')} / 카테고리: {etf_profile.get('category')}\n"
            f"투자 포커스: {etf_profile.get('focus')}\n"
            f"프로젝트 내 역할: {etf_profile.get('role_in_project')}\n"
            f"거시 민감도 요약: {etf_profile.get('macro_sensitivity')}\n"
            f"위험 프로필: {etf_profile.get('risk_profile')}\n"
            f"주요 섹터: {etf_profile.get('top_sectors')}\n"
            f"주요 지역: {etf_profile.get('top_regions')}\n"
            f"예시 편입 종목: {etf_profile.get('example_top_holdings')}\n"
        )

    # --- 가격 반응 요약 텍스트 ---
    reaction_lines = []
    for r in reactions[:8]:  # 너무 길어지는 걸 방지해서 상위 몇 개만
        # r이 dataclass이든 dict이든 둘 다 대응
        ticker = getattr(r, "ticker", None) or r.get("ticker", "")
        start_date = getattr(r, "start_date", None) or r.get("start_date", "")
        end_date = getattr(r, "end_date", None) or r.get("end_date", "")
        ret = getattr(r, "return_pct", None) or r.get("return_pct", 0.0)
        direction = getattr(r, "direction", None) or r.get("direction", "")

        reaction_lines.append(
            f"- {ticker} {start_date} ~ {end_date}: {float(ret):.2f}% ({direction})"
        )

    price_reactions_text = "\n".join(reaction_lines) if reaction_lines else "가격 반응 정보 없음"

    # --- 뉴스 요약 텍스트 ---
    news_lines = []
    for h in hits[:10]:
        title = (h.get("title") or "").strip()
        date = (str(h.get("date")) or "").strip()
        src = (h.get("source") or "").strip()
        txt = (h.get("text") or h.get("snippet") or "").strip()
        if len(txt) > 300:
            txt = txt[:300] + "..."
        news_lines.append(
            f"- [{date}] ({src}) {title}\n  {txt}"
        )

    news_text = "\n".join(news_lines) if news_lines else "관련 뉴스 없음"

    # --- (옵션) 거시 컨텍스트 텍스트 ---
    macro_text = ""
    if macro_context:
        # macro_context를 dict 형태로 받는다고 가정
        # 예: {"10Y_yield_change": -0.12, "VIX_change_pct": -4.3, ...}
        macro_lines = []
        if isinstance(macro_context, dict):
            for k, v in macro_context.items():
                macro_lines.append(f"- {k}: {v}")
            macro_text = "\n".join(macro_lines)
        else:
            macro_text = str(macro_context)

    prompt = f"""
당신은 ETF 가격과 뉴스 데이터를 함께 분석하는 증권사 리서치 센터 애널리스트입니다.
역할은 "뉴스·이벤트가 ETF 가격 움직임에 어떤 영향을 주었는지"를 설명하는 것입니다.

[사용자 질문]
{question}

[ETF 프로필]
{etf_profile_text or "ETF 구조 정보 없음"}

[가격 반응 요약]
{price_reactions_text}

[뉴스 요약]
{news_text}

[거시 환경 요약(있다면 참고)]
{macro_text or "거시 지표 요약 없음"}

요구사항:
1. 위 정보를 바탕으로, 최근 ETF 가격 움직임(상승/하락/변동성 확대 등)을 먼저 간단히 정리하세요.
2. 각 가격 반응 구간에 대해, 어떤 뉴스/이벤트가 주요한 촉매(catalyst)가 되었는지 연결해서 설명하세요.
   - ETF의 구조(섹터/자산군/테마)와 거시 민감도(금리, 유가, 위험선호 등)를 적극 활용하세요.
3. 인과관계를 지나치게 단정하지 말고, 
   - "직접적인 영향", "부분적으로 기여", "가능성이 있으나 불확실" 등으로 구분해 표현하세요.
4. 가격 반응에 비해 뉴스 강도가 과하거나 부족해 보이는 부분이 있으면, 
   - "가격이 선반영된 구간", "뉴스 대비 과민 반응", "기타 요인(예: 테마 수급, 파생상품 포지션)이 있었을 가능성" 등을 언급해도 좋습니다.
5. 전체 길이는 4~8문단 정도의 분석 리포트 느낌으로 작성하세요.

한국어로 작성해 주세요.
"""

    analysis = llm.invoke(prompt).content
    state["news_impact_analysis"] = analysis
    return state


ETF 프로필

In [ ]:
def attach_etf_profile(state: dict) -> dict:
    """
    matched_etfs에 있는 첫 번째 ETF에 대한 프로필 정보를 state에 추가.
    - etf_profile: {ticker, name, asset_class, category, ...}
    """
    tickers = state.get("matched_etfs", [])
    if not tickers:
        state["etf_profile"] = {}
        return state

    ticker = tickers[0]
    profile = etf_profile_table.get(ticker, {})
    state["etf_profile"] = profile
    return state

거시 데이터

In [ ]:
def load_macro_data(period: str = "2y") -> pd.DataFrame:
    dfs = []

    for name, ticker in MACRO_TICKERS.items():
        data = yf.download(ticker, period=period, auto_adjust=False, progress=False)
        if data.empty:
            continue

        df = data[["Close"]].copy()
        df.reset_index(inplace=True)
        df.sort_values("Date", inplace=True)  # ← 추가
        df.rename(columns={"Date": "date", "Close": f"{name}_close"}, inplace=True)
        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["date"])

    macro_df = dfs[0]

    for df in dfs[1:]:
        df.sort_values("date", inplace=True)                      # ← 추가
        macro_df.sort_values("date", inplace=True)                # ← 추가
        macro_df = macro_df.merge(df, on="date", how="outer")

    macro_df.sort_values("date", inplace=True)
    macro_df.reset_index(drop=True, inplace=True)

    # 변화율 계산 (fill_method=None 추가)
    for name in ["SPX", "VIX", "DXY", "GOLD", "BTC"]:
        col = f"{name}_close"
        if col in macro_df.columns:
            macro_df[f"{name}_ret_pct"] = macro_df[col].pct_change(fill_method=None) * 100

    if "UST10Y_close" in macro_df.columns:
        macro_df["UST10Y_change"] = macro_df["UST10Y_close"].diff()

    return macro_df


MACRO_TICKERS = {
    "SPX": "^GSPC",     # S&P 500
    "VIX": "^VIX",      # VIX index
    "UST10Y": "^TNX",   # 10Y 국채 (단위: % * 100)
    "DXY": "DX-Y.NYB",  # Dollar Index
    "GOLD": "GC=F",     # COMEX Gold Futures (금 가격 대표)
    "BTC": "BTC-USD"    # Bitcoin Spot (달러 기준)
}

# 전역 거시 데이터프레임
macro_df = load_macro_data(period="2y")
print("macro_df loaded, rows:", len(macro_df))

유틸 함수

In [ ]:
def now_kst_date_str() -> str:
    kst = timezone(timedelta(hours=9))
    return datetime.now(kst).strftime("%Y-%m-%d")

def _now_kst_str(fmt="%Y%m%d_%H%M%S"):
    kst = pytz.timezone("Asia/Seoul")
    return datetime.now(kst).strftime(fmt)

def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def boost_query_keywords(
    hits: List[Dict[str, Any]],
    query: str,
    boost: float = 0.1,
) -> List[Dict[str, Any]]:
    """
    사용자가 입력한 질의(query)에 등장하는 키워드가
    기사 제목/본문에 포함될수록 score를 올려주는 함수.

    - boost: 최대 몇 %까지 올릴지 (0.1 => 최대 10% 가산 느낌)
    """
    query = (query or "").strip()
    if not query:
        return hits

    # 질의 토큰화 (한/영/숫자)
    q_tokens = tokenize_ko(query)
    q_tokens = [t for t in q_tokens if len(t) >= 2]  # 한 글자짜리 제거
    if not q_tokens:
        return hits

    q_token_set = set(q_tokens)

    new_hits = []
    for h in hits:
        base_score = float(h.get("score", 0.0) or 0.0)

        # doc 기반인지 / web 기반인지에 따라 텍스트 추출
        doc = h.get("doc")
        if doc is not None:
            text = (doc.metadata.get("title", "") + "\n" + doc.page_content)
        else:
            text = (h.get("title", "") + "\n" + h.get("text", ""))

        t_tokens = tokenize_ko(text)
        t_token_set = set(t_tokens)

        matched = q_token_set & t_token_set
        if matched:
            # 질의 토큰 중 몇 개가 매칭되는지 비율
            ratio = len(matched) / len(q_token_set)
            factor = 1.0 + boost * ratio
            new_score = base_score * factor
        else:
            new_score = base_score

        h2 = dict(h)
        h2["score"] = new_score
        new_hits.append(h2)

    new_hits.sort(key=lambda x: x.get("score", 0.0), reverse=True)
    return new_hits

def _rq(state: GraphState) -> str:
    rq = state.get("rewritten_question")
    if rq and rq.strip():
        return rq.strip()
    return (state.get("question") or "").strip()

def _to_naive(dt):
    try:
        return pd.to_datetime(dt, utc=True).tz_localize(None)
    except Exception:
        return pd.to_datetime(dt).tz_localize(None) if hasattr(pd.to_datetime(dt), "tz_localize") else pd.to_datetime(dt)
    

# Price window implementation is defined once in cell 16 and price_windows.py.

def _passthrough(prev_state: dict, new_writes: dict, keys=("price_context","etf_stats")) -> dict:
    """new_writes에 해당 키가 비어 있으면 prev_state의 기존 값을 유지한다."""
    out = dict(new_writes)
    for k in keys:
        if k in new_writes:
            v = new_writes[k]
            is_blank = (
                v is None or
                (isinstance(v, str) and v.strip() == "") or
                (isinstance(v, dict) and len(v) == 0) or
                (isinstance(v, list) and len(v) == 0)
            )
            if is_blank:
                out[k] = prev_state.get(k, v)
    return out


def _near(a, b, days=1):
    return abs((pd.to_datetime(a) - pd.to_datetime(b)).days) <= days

def _parse_date(s: str):
    if isinstance(s, str):
        for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y-%m-%dT%H:%M:%S"):
            try:
                return datetime.strptime(s, fmt).date()
            except ValueError:
                continue
    return None

def save_report_md(
    state: Dict[str, Any],
    out_dir: str = "reports",
    prefix: str = "etf_report",
    fig_paths: List[str] = None,
) -> str:
    """
    LangGraph 최종 state + 생성된 이미지들을 이용해
    자동 분석 리포트(Markdown)를 생성한다.

    Parameters
    ----------
    state : dict
        graph.invoke() 결과 전체 state
    out_dir : str
        md 파일 저장 폴더
    prefix : str
        파일 이름 prefix
    fig_paths : List[str]
        save_visuals() 에서 생성된 이미지 경로들

    Returns
    -------
    md_path : str
        저장된 markdown 파일 경로
    """
    os.makedirs(out_dir, exist_ok=True)

    final_answer = state.get("final_answer", "")
    question = state.get("question", "")

    # state["today"]가 있으면 쓰고, 없으면 오늘 날짜 사용
    raw_today = state.get("today")
    if raw_today:
        today = str(raw_today)
    else:
        today = str(_dt.date.today())

    price_context = state.get("price_context", "")
    etf_stats = state.get("etf_stats", {})

    if fig_paths is None:
        fig_paths = []

    # 이미지 그룹 분류
    recent_imgs = [p for p in fig_paths if "price_recent" in p]
    compare_imgs = [p for p in fig_paths if "normalized_compare" in p]
    event_imgs = [p for p in fig_paths if "_event_" in p and "overlay" not in p]
    overlay_imgs = [p for p in fig_paths if "event_overlay" in p]
    hybrid_imgs = [p for p in fig_paths if "hybrid" in p]

    md_lines = []

    # ==========================================================
    # Header
    # ==========================================================
    md_lines.append(f"# 📈 ETF 영향 분석 리포트\n")
    md_lines.append(f"- 질문: **{question}**")
    md_lines.append(f"- 분석 기준일: **{today}**\n")

    # ==========================================================
    # Summary (final_answer)
    # ==========================================================
    md_lines.append("\n---\n")
    md_lines.append("## 📝 분석 요약\n")
    md_lines.append(final_answer)
    md_lines.append("\n")

    # ==========================================================
    # Price context
    # ==========================================================
    md_lines.append("\n---\n")
    md_lines.append("## 💡 가격 컨텍스트\n")
    md_lines.append(f"```\n{price_context}\n```")

    # ==========================================================
    # Recent price charts
    # ==========================================================
    if recent_imgs:
        md_lines.append("\n---\n")
        md_lines.append("## 📊 최근 가격 추이 (최근 60 거래일)\n")
        for p in recent_imgs:
            rel = os.path.relpath(p, out_dir).replace("\\", "/")
            md_lines.append(f"![recent]({rel})\n")

    # ==========================================================
    # Compare (normalized) charts
    # ==========================================================
    if compare_imgs:
        md_lines.append("\n---\n")
        md_lines.append("## 📈 ETF 정규화 비교 (최근 60거래일)\n")
        for p in compare_imgs:
            rel = os.path.relpath(p, out_dir).replace("\\", "/")
            md_lines.append(f"![compare]({rel})\n")

    # ==========================================================
    # Event windows (개별)
    # ==========================================================
    if event_imgs:
        md_lines.append("\n---\n")
        md_lines.append("## 📉 과거 유사 뉴스 이벤트 윈도우 (개별)\n")
        for p in event_imgs:
            rel = os.path.relpath(p, out_dir).replace("\\", "/")
            md_lines.append(f"![event]({rel})\n")

    # ==========================================================
    # Overlay
    # ==========================================================
    if overlay_imgs:
        md_lines.append("\n---\n")
        md_lines.append("## 🪄 이벤트 윈도우 Overlay (여러 이벤트 비교)\n")
        for p in overlay_imgs:
            rel = os.path.relpath(p, out_dir).replace("\\", "/")
            md_lines.append(f"![overlay]({rel})\n")

    # ==========================================================
    # Hybrid
    # ==========================================================
    if hybrid_imgs:
        md_lines.append("\n---\n")
        md_lines.append("## 🔀 하이브리드: 최근 추이 + 이벤트 패턴\n")
        for p in hybrid_imgs:
            rel = os.path.relpath(p, out_dir).replace("\\", "/")
            md_lines.append(f"![hybrid]({rel})\n")

    # ==========================================================
    # Save MD
    # ==========================================================
    md_name = f"{prefix}_{today}.md"
    md_path = os.path.join(out_dir, md_name)

    with open(md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(md_lines))

    return md_path


def save_run_logs(state: dict, out_dir: str = "logs", prefix: str = "run"):
    ''' 
    로그 저장 (reranked CSV / etf_stats JSON / price_context TXT / run_meta JSON)
    '''


    _ensure_dir(out_dir)
    ts = _now_kst_str()

    # 1) reranked_hits → CSV
    reranked = state.get("reranked_hits", []) or []
    csv_path = os.path.join(out_dir, f"reranked_{ts}.csv")
    if reranked:
        # 키 정규화
        df = pd.DataFrame(reranked)
        # 열 순서(가독성)
        cols_pref = ["id","doc_id","title","date","source","url","origin","score",
                     "score_bm25","score_dense","score_web","text"]
        cols = [c for c in cols_pref if c in df.columns] + [c for c in df.columns if c not in cols_pref]
        df = df[cols]
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    else:
        csv_path = None

    # 2) etf_stats → JSON
    etf_stats = state.get("etf_stats", {}) or {}
    etf_json_path = os.path.join(out_dir, f"etf_stats_{ts}.json")
    with open(etf_json_path, "w", encoding="utf-8") as f:
        json.dump(etf_stats, f, ensure_ascii=False, indent=2)

    # 3) price_context → TXT
    price_ctx = state.get("price_context", "") or ""
    price_txt_path = os.path.join(out_dir, f"price_context_{ts}.txt")
    with open(price_txt_path, "w", encoding="utf-8") as f:
        f.write(price_ctx)

    # 4) 실행 메타 → JSON
    meta = {
        "created_at_kst": _now_kst_str("%Y-%m-%d %H:%M:%S"),
        "question": state.get("question", ""),
        "counts": {
            "bm25": len(state.get("bm25_hits", []) or []),
            "dense": len(state.get("dense_hits", []) or []),
            "web": len(state.get("web_hits", []) or []),
            "fused": len(state.get("fused_hits", []) or []),
            "reranked": len(reranked),
        },
        "matched_etfs": state.get("matched_etfs", []),
        "consistency_scores": state.get("consistency_scores", {}),
        "config": {
            "ensemble_topN": getattr(CFG, "ensemble_topN", None),
            "rerank_topk": getattr(CFG, "rerank_topk", None),
            "rrf_k": getattr(CFG, "rrf_k", None),
            "embedding_backend": getattr(CFG, "embedding_backend", None),
            "reranker_backend": getattr(CFG, "reranker_backend", None),
            "use_naver": getattr(CFG, "use_naver", None),
            "thread_id": getattr(CFG, "thread_id", None),
        },
    }
    meta_path = os.path.join(out_dir, f"{prefix}_meta_{ts}.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return {
        "reranked_csv": csv_path,
        "etf_stats_json": etf_json_path,
        "price_context_txt": price_txt_path,
        "meta_json": meta_path,
    }


def save_visuals(
    state: Dict[str, Any],
    price_df: pd.DataFrame,
    out_dir: str = "reports/figs",
    max_etfs: int = 5,
    recent_days: int = 60,
    # 🔸 이벤트 스터디 옵션
    event_pre_days: int = 0,   # 이벤트 기준일 이전 며칠까지 볼지 (lead)
    event_post_days: int = 3,  # 이벤트 기준일 이후 며칠까지 볼지 (lag)
    make_event_overlay: bool = True,
    make_event_hybrid: bool = True,
) -> List[str]:
    """
    LangGraph state + 가격 데이터(price_df)를 받아
    1) 개별 ETF 최근 N일 차트
    2) 여러 ETF 정규화 비교 차트
    3) 유사 기사 시점별 이벤트 윈도우 차트 (각 윈도우 개별)
    4) 유사 기사 시점별 이벤트 윈도우 overlay (relative day 기준)
    5) 최근 구간 + 이벤트 overlay 를 함께 보는 hybrid 차트
    를 PNG로 저장하고, 파일 경로 리스트를 반환한다.

    Parameters
    ----------
    state : dict
        graph.invoke(...) 의 결과 state.
        state["matched_etfs"], state["etf_stats"] 사용.
    price_df : pd.DataFrame
        최소한 다음 컬럼을 포함해야 한다:
        - 'date_kst' : 날짜 (datetime 또는 문자열)
        - 'ticker'   : ETF 티커 문자열
        - 가격 컬럼  : 'close' 또는 ['adj_close', 'Close', 'close_price'] 중 하나
    out_dir : str
        이미지를 저장할 디렉토리 경로.
    max_etfs : int
        최대 몇 개 ETF까지 시각화할지.
    recent_days : int
        최근 몇 영업일(행) 데이터까지 가져와서 그림을 그릴지.
    event_pre_days : int
        이벤트 기준일(event_day) 이전 며칠까지 볼지 (리드).
    event_post_days : int
        이벤트 기준일 이후 며칠까지 볼지 (래그).

    Returns
    -------
    fig_paths : List[str]
        생성된 모든 이미지 파일 경로 리스트.
    """
    os.makedirs(out_dir, exist_ok=True)

    df = price_df.copy()

    # --- date_kst를 datetime으로 정규화 ---
    if not pd.api.types.is_datetime64_any_dtype(df["date_kst"]):
        df["date_kst"] = pd.to_datetime(df["date_kst"])

    df = df.sort_values(["ticker", "date_kst"])

    # --- 가격 컬럼 결정 ---
    value_col = None
    candidate_cols = ["close", "adj_close", "Close", "close_price"]
    for c in candidate_cols:
        if c in df.columns:
            value_col = c
            break

    if value_col is None:
        raise ValueError(
            f"price_df에 가격 컬럼({candidate_cols}) 중 하나가 필요합니다."
        )

    # --- 대상 ETF 리스트 결정 ---
    matched_etfs = state.get("matched_etfs") or []
    if not matched_etfs:
        # state에 없으면 데이터에서 상위 몇 개 티커 자동 선택
        matched_etfs = list(df["ticker"].dropna().unique())[:max_etfs]
    else:
        matched_etfs = matched_etfs[:max_etfs]

    fig_paths: List[str] = []

    # ==========================================================
    # 1) 개별 ETF 최근 N일 가격 라인 차트
    # ==========================================================
    for ticker in matched_etfs:
        sub = df[df["ticker"] == ticker].sort_values("date_kst")
        if sub.empty:
            continue

        # 최근 recent_days 행만 사용
        sub_recent = sub.tail(recent_days)
        if sub_recent.empty:
            continue

        plt.figure(figsize=(10, 4))
        plt.plot(sub_recent["date_kst"], sub_recent[value_col])
        plt.title(f"{ticker} 가격 (최근 {len(sub_recent)}거래일)")
        plt.xlabel("날짜")
        plt.ylabel("가격")
        plt.xticks(rotation=30)
        plt.tight_layout()

        fpath = os.path.join(out_dir, f"{ticker}_price_recent.png")
        plt.savefig(fpath, dpi=150)
        plt.close()
        fig_paths.append(fpath)

    # ==========================================================
    # 2) 여러 ETF를 한 그래프에: 정규화 수익률(=처음을 1로 맞춘 가격)
    # ==========================================================
    valid_etfs = []
    for t in matched_etfs:
        if (df["ticker"] == t).any():
            valid_etfs.append(t)

    if len(valid_etfs) >= 2:
        # 전체 데이터에서 가장 최근 날짜 기준으로 recent_days만큼 자른다.
        last_date = df["date_kst"].max()
        start_date = last_date - pd.Timedelta(days=recent_days * 2)

        merged_list = []
        for ticker in valid_etfs:
            sub = df[
                (df["ticker"] == ticker)
                & (df["date_kst"] >= start_date)
            ].sort_values("date_kst")

            if sub.empty:
                continue

            sub_recent = sub.tail(recent_days)
            if sub_recent.empty:
                continue

            first_val = sub_recent[value_col].iloc[0]
            if first_val == 0:
                continue

            sub_recent = sub_recent.copy()
            sub_recent["norm_price"] = sub_recent[value_col] / first_val
            sub_recent["ticker"] = ticker
            merged_list.append(sub_recent[["date_kst", "ticker", "norm_price"]])

        if merged_list:
            merged = pd.concat(merged_list, ignore_index=True)

            plt.figure(figsize=(10, 5))
            for ticker in merged["ticker"].unique():
                subm = merged[merged["ticker"] == ticker].sort_values("date_kst")
                plt.plot(subm["date_kst"], subm["norm_price"], label=ticker)

            plt.title(f"ETF 비교 (정규화 가격, 최근 {recent_days}거래일)")
            plt.xlabel("날짜")
            plt.ylabel("정규화 가격 (시작=1)")
            plt.xticks(rotation=30)
            plt.legend()
            plt.tight_layout()

            fpath = os.path.join(out_dir, f"etf_normalized_compare.png")
            plt.savefig(fpath, dpi=150)
            plt.close()
            fig_paths.append(fpath)

    # ==========================================================
    # 3) 유사 기사 시점별 이벤트 윈도우 차트 (hist_formatted 기반, 각 윈도우 개별)
    # ==========================================================
    etf_stats: Dict[str, Any] = state.get("etf_stats", {}) or {}
    date_range_pat = re.compile(r"(\d{4}-\d{2}-\d{2})~(\d{4}-\d{2}-\d{2})")

    # overlay / hybrid에서 다시 쓰기 위해, 윈도우 정보를 모아둔다.
    event_windows_by_ticker: Dict[str, List[Dict[str, Any]]] = {t: [] for t in matched_etfs}

    for ticker in matched_etfs:
        stats = etf_stats.get(ticker) or {}
        hist_formatted = stats.get("hist_formatted") or []
        if not hist_formatted:
            continue

        for idx, s in enumerate(hist_formatted, start=1):
            m = date_range_pat.search(str(s))
            if not m:
                continue

            start_str, end_str = m.group(1), m.group(2)
            start_dt = pd.to_datetime(start_str)
            end_dt = pd.to_datetime(end_str)

            # 개별 윈도우: 원래 정의된 window 그대로 사용
            sub = df[
                (df["ticker"] == ticker)
                & (df["date_kst"] >= start_dt)
                & (df["date_kst"] <= end_dt)
            ].sort_values("date_kst")

            if sub.empty:
                continue

            first_val = sub[value_col].iloc[0]
            if first_val == 0:
                continue

            sub = sub.copy()
            sub["norm_price"] = sub[value_col] / first_val

            label = f"과거 유사 기사 {idx}"

            plt.figure(figsize=(8, 3.5))
            plt.plot(sub["date_kst"], sub["norm_price"])
            plt.title(f"{ticker} – {label}\n({start_dt.date()}~{end_dt.date()})")
            plt.xlabel("날짜")
            plt.ylabel("정규화 가격 (윈도우 시작=1)")
            plt.xticks(rotation=30)
            plt.tight_layout()

            fpath = os.path.join(
                out_dir,
                f"{ticker}_event_{idx}_{start_dt.date()}_{end_dt.date()}.png",
            )
            plt.savefig(fpath, dpi=150)
            plt.close()
            fig_paths.append(fpath)

            # overlay / hybrid용으로 정보 저장
            event_windows_by_ticker[ticker].append(
                {
                    "start": start_dt,
                    "end": end_dt,
                    "label": label,
                    "index": idx,
                }
            )

    # ==========================================================
    # 4) 이벤트 윈도우 overlay (relative day 기준, 리드/래그 포함)
    # ==========================================================
    if make_event_overlay:
        for ticker in matched_etfs:
            windows = event_windows_by_ticker.get(ticker) or []
            if len(windows) < 2:
                continue  # overlay는 최소 2개 이상일 때 의미 있음

            plt.figure(figsize=(9, 4))

            for w in windows:
                event_start = w["start"]  # 이벤트 기준일(기사 날짜)로 사용
                start_dt = event_start - pd.Timedelta(days=event_pre_days)
                end_dt = event_start + pd.Timedelta(days=event_post_days)

                sub = df[
                    (df["ticker"] == ticker)
                    & (df["date_kst"] >= start_dt)
                    & (df["date_kst"] <= end_dt)
                ].sort_values("date_kst")

                if sub.empty:
                    continue

                first_val = sub[value_col].iloc[0]
                if first_val == 0:
                    continue

                sub = sub.copy()
                sub["norm_price"] = sub[value_col] / first_val
                # relative day: 이벤트 기준일 = 0
                sub["rel_day"] = (sub["date_kst"] - event_start).dt.days

                plt.plot(sub["rel_day"], sub["norm_price"],
                         label=f"{w['label']} ({event_start.date()})")

            plt.axvline(0, linestyle="--")  # 이벤트 기준일 표시
            plt.title(
                f"{ticker} 이벤트 윈도우 overlay\n"
                f"(기준일±[{event_pre_days}, {event_post_days}]일)"
            )
            plt.xlabel("이벤트 기준일로부터 경과 일수 (일)")
            plt.ylabel("정규화 가격 (시작=1)")
            plt.tight_layout()
            plt.legend()

            fpath = os.path.join(
                out_dir,
                f"{ticker}_event_overlay_{event_pre_days}_{event_post_days}.png",
            )
            plt.savefig(fpath, dpi=150)
            plt.close()
            fig_paths.append(fpath)

    # ==========================================================
    # 5) 하이브리드 차트: 최근 구간 + 이벤트 overlay 를 함께 보기
    #    (subplots 2개: 위=최근 N일, 아래=relative overlay)
    # ==========================================================
    if make_event_hybrid:
        for ticker in matched_etfs:
            windows = event_windows_by_ticker.get(ticker) or []
            if not windows:
                continue

            sub_all = df[df["ticker"] == ticker].sort_values("date_kst")
            if sub_all.empty:
                continue

            sub_recent = sub_all.tail(recent_days)
            if sub_recent.empty:
                continue

            # overlay용 준비: 4번에서 했던 것과 거의 동일
            overlay_segments = []
            for w in windows:
                event_start = w["start"]
                start_dt = event_start - pd.Timedelta(days=event_pre_days)
                end_dt = event_start + pd.Timedelta(days=event_post_days)

                sub = df[
                    (df["ticker"] == ticker)
                    & (df["date_kst"] >= start_dt)
                    & (df["date_kst"] <= end_dt)
                ].sort_values("date_kst")

                if sub.empty:
                    continue

                first_val = sub[value_col].iloc[0]
                if first_val == 0:
                    continue

                sub = sub.copy()
                sub["norm_price"] = sub[value_col] / first_val
                sub["rel_day"] = (sub["date_kst"] - event_start).dt.days
                overlay_segments.append((w, sub))

            if not overlay_segments:
                continue

            # 하이브리드 플롯
            fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=False)

            # (위) 최근 구간
            axes[0].plot(sub_recent["date_kst"], sub_recent[value_col])
            axes[0].set_title(f"{ticker} 최근 {len(sub_recent)}거래일 가격")
            axes[0].set_xlabel("날짜")
            axes[0].set_ylabel("가격")
            for label in axes[0].get_xticklabels():
                label.set_rotation(30)

            # (아래) overlay
            for (w, sub) in overlay_segments:
                axes[1].plot(
                    sub["rel_day"],
                    sub["norm_price"],
                    label=f"{w['label']} ({w['start'].date()})",
                )
            axes[1].axvline(0, linestyle="--", alpha=0.7)
            axes[1].set_title(
                f"{ticker} 이벤트 윈도우 overlay\n"
                f"(기준일±[{event_pre_days}, {event_post_days}]일)"
            )
            axes[1].set_xlabel("이벤트 기준일로부터 경과 일수 (일)")
            axes[1].set_ylabel("정규화 가격 (시작=1)")
            axes[1].legend()
            for label in axes[1].get_xticklabels():
                label.set_rotation(0)

            plt.tight_layout()

            fpath = os.path.join(
                out_dir,
                f"{ticker}_hybrid_recent_and_events.png",
            )
            plt.savefig(fpath, dpi=150)
            plt.close(fig)
            fig_paths.append(fpath)

    return fig_paths

## 네이버 크롤링 함수

In [ ]:
def _strip_tags(s: str) -> str:
    if not s:
        return ""
    # 네이버는 <b>태그/HTML 엔티티가 섞여있음
    s = html.unescape(s)
    s = _TAG_RE.sub("", s)
    return s.strip()

def _to_iso_utc(pubdate_str: str) -> str:
    """
    Naver pubDate 예: 'Tue, 05 Nov 2025 13:21:00 +0900'
    → ISO UTC 문자열로 변환 (파이프라인에서 utc=True로 파싱하기 쉬움)
    """
    try:
        dt_kst = pd.to_datetime(pubdate_str, utc=True)  # +0900 포함되어 있어 바로 tz-aware
        return dt_kst.isoformat()  # '2025-11-05T04:21:00+00:00'
    except Exception:
        return None

def _naver_news_search(query: str, display: int, start: int, sort: str = "date") -> Dict[str, Any]:
    headers = {
        "X-Naver-Client-Id": CFG.naver_client_id,
        "X-Naver-Client-Secret": CFG.naver_client_secret,
    }
    params = {
        "query": query,
        "display": display,   # 한 페이지 개수
        "start": start,       # 페이지 시작 index (1, 11, 21, ...)
        "sort": sort,         # date | sim
    }
    resp = requests.get(NAVER_NEWS_ENDPOINT, headers=headers, params=params, timeout=CFG.naver_timeout)
    resp.raise_for_status()
    return resp.json()

def _normalize_naver_items(items: List[Dict[str, Any]], source_label: str = "web") -> List[Dict[str, Any]]:
    """
    파이프라인 표준 스키마로 맞춤:
    id/doc_id, title, text, date, source, url, origin, score_web
    """
    out = []
    for i, it in enumerate(items):
        title = _strip_tags(it.get("title", ""))
        desc  = _strip_tags(it.get("description", ""))
        link  = it.get("link") or it.get("originallink")
        date  = _to_iso_utc(it.get("pubDate"))
        # 간단한 랭크 기반 점수(최신/상위 우선). 필요 시 리랭커가 덮어씀.
        score_web = max(0.0, 1.0 - 0.02 * i)

        out.append({
            "id": f"W{i+1}",               # 내부 id
            "doc_id": f"W{i+1}",           # 너가 쓰던 필드 호환
            "title": title,
            "text": desc,
            "date": date,                  # ISO UTC (예: '2025-11-05T04:21:00+00:00')
            "source": source_label,
            "url": link,
            "origin": "web",               # fused 단계에서 구분용
            "score_web": score_web,
            # (옵션) 다른 스코어 자리도 미리 0으로
            "score_bm25": 0.0,
            "score_dense": 0.0,
            "score": score_web,
        })
    return out

def filter_today_kst(web_hits):
    KST = pytz.timezone("Asia/Seoul")
    today_kst = datetime.now(KST).date()
    out = []
    for h in web_hits:
        dt = pd.to_datetime(h.get("date"), utc=True, errors="coerce")
        if pd.isna(dt): 
            continue
        if dt.tzinfo is None:  # 혹시 모를 naive
            dt = dt.tz_localize(timezone.utc)
        if dt.astimezone(KST).date() == today_kst:
            out.append(h)
    return out

## 노드 stubs

In [ ]:

def init_today(state: GraphState) -> GraphState:
    new_state = dict(state)
    cutoff = market_timestamp(state.get("as_of") or datetime.now(timezone.utc).isoformat())
    new_state["as_of"] = cutoff.isoformat()
    new_state["today"] = cutoff.tz_convert("Asia/Seoul").strftime("%Y-%m-%d")
    return new_state

def rewrite_query(state: GraphState) -> GraphState:
    q = state.get("question", "").strip()
    if not q:
        return {"rewritten_question": ""}  # 빈 값이라도 써 둠(가드용)

    prompt = f"사용자 질문을 경제 뉴스 검색 키워드로 짧게 재작성:\n- 원문: {q}\n- 재작성(키워드):"
    try:
        out = llm.invoke(prompt).content.strip()
    except Exception:
        out = q  # 실패시 원문 사용

    return {"rewritten_question": out}

def retrieve_bm25(state) -> Dict[str, Any]:
    """BM25로 로컬 뉴스 검색 → 상위 N개 반환"""
    query = state.get("rewritten_query") or state.get("question") or ""
    if not query.strip():
        return {"bm25_hits": []}

    q_tokens = tokenize_ko(query)
    scores = _BM25.get_scores(q_tokens)  # len == len(LOCAL_DOCS)

    # 상위 N개 뽑기
    TOPN = 20
    idx_sorted = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:TOPN]
    max_s = max(scores[i] for i in idx_sorted) or 1.0
    # 0~1 정규화
    hits = []

    rq = _rq(state)  # ✅ 안전
    if not rq:
        return {"web_hits": []}
    
    for i in idx_sorted:
        d = LOCAL_DOCS[i]
        norm = scores[i] / max_s
        hits.append(_doc_to_hit(d, norm, origin="bm25"))
    return {"bm25_hits": hits}

def retrieve_dense(state) -> Dict[str, Any]:
    """벡터 스토어 의미검색 → 상위 N개 반환"""
    query = state.get("rewritten_query") or state.get("question") or ""
    if not query.strip():
        return {"dense_hits": []}

    TOPK = 20
    try:
        # vectorstore.similarity_search_with_score 권장
        results = vectorstore.similarity_search_with_score(query, k=TOPK)
        # 결과 형태: List[Tuple[Document, score]]
        hits = []
        for d, s in results:
            hits.append(_doc_to_hit(d, _norm_dense_score(float(s)), origin="dense"))
    except Exception:
        # fallback
        docs = vectorstore.similarity_search(query, k=TOPK)
        hits = [_doc_to_hit(d, 0.5, origin="dense") for d in docs]

    rq = _rq(state)  # ✅ 안전
    if not rq:
        return {"web_hits": []}
    
    return {"dense_hits": hits}

def _norm_dense_score(raw_score: float) -> float:
    """
    다양한 벡터DB가 점수를 다르게 주므로 범용 정규화:
    - 0~1 유사도면 그대로 사용
    - 거리(작을수록 유사): 1/(1+dist)
    - 음수/기타는 시그모이드로 압축
    """
    if 0.0 <= raw_score <= 1.0:
        return float(raw_score)
    if raw_score > 1.0:
        return float(1.0 / (1.0 + raw_score))
    # 음수 등 기타 스케일
    return float(1.0 / (1.0 + math.exp(-raw_score)))

def retrieve_web(state: GraphState) -> GraphState:
    """
    네이버 오늘 뉴스 검색/수집 (금일, 경제/증권 섹션).
    - rewritten_question 우선, 없으면 question 사용
    - 최신순(sort='date')으로 페이징 수집
    """
    rq = (state.get("rewritten_question") or state.get("question") or "").strip()
    if not rq or not CFG.use_naver:
        return {"web_hits": []}

    all_items = []
    start_idx = 1
    for page in range(CFG.naver_max_pages):
        try:
            data = _naver_news_search(
                query=rq,
                display=CFG.naver_page_size,
                start=start_idx,
                sort="date"   # 최신순
            )
        except Exception as e:
            print(f"[WARN] Naver API error: {e}")
            break

        items = data.get("items", []) or []
        if not items:
            break

        all_items.extend(items)

        # 다음 페이지 시작 인덱스 (네이버는 start를 1, 11, 21... 로 줌)
        start_idx += CFG.naver_page_size
        # 과도한 호출 방지
        time.sleep(0.15)

    # 정규화
    web_hits = _normalize_naver_items(all_items, source_label="web")

    # 간단 중복 제거(제목+URL 기준)
    seen = set()
    uniq = []
    for h in web_hits:
        k = (h["title"], h.get("url"))
        if k in seen: 
            continue
        seen.add(k)
        uniq.append(h)
    web_hits = _normalize_naver_items(all_items, source_label="web")
    web_hits = filter_today_kst(web_hits)   # ← 오늘 기사만 필요시 ON
    return {"web_hits": web_hits}


def fuse_results(state: GraphState) -> GraphState:
    bm25 = state.get("bm25_hits", [])
    dense = state.get("dense_hits", [])
    web   = state.get("web_hits", [])
    pool = []

    # 각 소스 RRF 점수 부여 (기존 로직 유지)
    def _rrf(items, score_key):
        scored = []
        for rank, it in enumerate(items, start=1):
            it = dict(it)
            it.setdefault("origin", score_key.replace("score_", ""))  # web/bm25/dense
            it.setdefault("score_bm25", 0.0)
            it.setdefault("score_dense", 0.0)
            it.setdefault("score_web", 0.0)
            # rrf 점수 누적
            rrf = 1.0 / (CFG.rrf_k + rank)
            it[score_key] = it.get(score_key, 0.0) + rrf
            it["score"] = it.get("score_bm25",0)+it.get("score_dense",0)+it.get("score_web",0)
            scored.append(it)
        return scored

    pool.extend(_rrf(bm25, "score_bm25"))
    pool.extend(_rrf(dense, "score_dense"))
    pool.extend(_rrf(web,   "score_web"))

    # 중복 제거(제목+url) 후 상위 N
    seen, uniq = set(), []
    for d in sorted(pool, key=lambda x: x.get("score",0), reverse=True):
        k = (d.get("title"), d.get("url"))
        if k in seen: 
            continue
        seen.add(k)
        uniq.append(d)
        if len(uniq) >= CFG.ensemble_topN:
            break

    return {"fused_hits": uniq}

def _hit_text(h: Dict[str, Any]) -> str:
    """
    리랭킹에 쓸 텍스트 하나로 합치기:
    - text (본문) > snippet > title 우선순위
    """
    txt = (h.get("text") or "").strip()
    if not txt:
        txt = (h.get("snippet") or "").strip()
    if not txt:
        txt = (h.get("title") or "").strip()
    return txt

def rerank_results(state) -> dict:
    """
    bge-m3(BGEM3FlagModel)를 사용해 fuse_results의 상위 문서를 재정렬하는 단계.
    반환: {"reranked_hits": [...]}  ← 이후 단계에서 이 key를 사용.
    """
    hits = state.get("fused_hits", []) or []
    if not hits:
        return {"reranked_hits": []}

    # === 0) 사용자 질의 기반 키워드 부스팅 (기존 로직 유지) ===
    query = _rq(state) or ""  # rewritten_question 우선, 없으면 question
    hits = boost_query_keywords(hits, query=query, boost=0.1)

    # 후보풀 크기 제한 – 너무 많으면 리랭킹 속도/비용이 커지므로 상한 적용
    hits = hits[:CFG.ensemble_topN]

    # === 1) bge-m3 입력용 [query, 문서텍스트] 페어 만들기 ===
    pairs = []
    valid_indices = []  # hits 인덱스를 따로 저장
    for idx, h in enumerate(hits):
        doc_text = _hit_text(h)
        if not doc_text:
            continue
        pairs.append([query, doc_text])
        valid_indices.append(idx)

    if not pairs:
        # 텍스트가 하나도 없으면 그냥 상위 k개를 그대로 사용
        base = hits[:CFG.rerank_topk]
    else:
        # === 2) bge-m3로 relevance score 계산 ===
        # compute_score: [[query, doc], ...] -> List[float]
        scores = bge_m3.compute_score(
            pairs,
            max_passage_length=256,   # 문서가 너무 길면 잘라서 사용
            # weights_for_different_modes=[0.4, 0.2, 0.4],  # dense/sparse/colbert 비율, 원하면 튜닝
        )

        # valid_indices와 scores를 묶어서 점수 기준 내림차순 정렬
        scored = sorted(
            zip(valid_indices, scores),
            key=lambda x: x[1],
            reverse=True,
        )

        # 상위 rerank_topk개 인덱스 선택
        top_indices = [i for i, _ in scored[:CFG.rerank_topk]]
        base = [hits[i] for i in top_indices]

    # 만약 base가 비어버리면(극단적인 경우) 단순 상위 k로 fallback
    if not base:
        base = hits[:CFG.rerank_topk]

    reranked = list(base)

    # === 3) 필드 보강 (빈 값 방지, 이후 단계에서 안전하게 쓰기 위해) ===
    for h in reranked:
        h.setdefault("title", "")
        h.setdefault("text", "")
        h.setdefault("date", None)
        h.setdefault("source", "unknown")
        h.setdefault("url", None)

    return {"reranked_hits": reranked}


def final_answer(state: dict) -> dict:
    """
    최종 인사이트 생성 노드.
    - 가격 요약(price_context)
    - ETF 기본 정보(etf_stats)
    - ETF 프로필(etf_profile)
    - 뉴스 영향 분석(news_impact_analysis)
    - 뉴스-가격 일관성(consistency_score/comment)
    - (옵션) 거시 컨텍스트(macro_context)
    를 모두 결합해, 최종 리포트 형식의 답변을 생성한다.
    """

    question = state.get("question", "")
    if not state.get("etf_stats"):
        return {"final_answer": "가격 관측 구간이 완료되지 않아 영향 분석을 생성하지 않았습니다.\n"
                + f"관측 기준시각: {state.get('as_of', '미지정')}\n"
                + state.get("price_context", "가격 정보 없음")}

    price_ctx = state.get("price_context", "") or ""
    etf_stats = state.get("etf_stats", {}) or {}
    etf_profile = state.get("etf_profile", {}) or {}
    impact = state.get("news_impact_analysis", "") or ""

    macro_context = state.get("macro_context", None)

    # 일관성 점수/코멘트
    consistency_score = (
        state.get("consistency_score")
        or state.get("price_news_consistency_score")
        or state.get("consistency")
    )
    consistency_comment = (
        state.get("consistency_comment")
        or state.get("price_news_consistency_explanation")
        or state.get("consistency_explanation")
        or ""
    )

    # etf_stats 텍스트 변환
    if isinstance(etf_stats, dict) and etf_stats:
        stats_lines = [f"- {k}: {v}" for k, v in etf_stats.items()]
        etf_stats_text = "\n".join(stats_lines)
    else:
        etf_stats_text = str(etf_stats) if etf_stats else "ETF 통계 정보 없음"

    # etf_profile 텍스트 변환 (요약 버전)
    etf_profile_text = ""
    if etf_profile:
        etf_profile_text = (
            f"티커: {etf_profile.get('ticker')}, 이름: {etf_profile.get('name')}\n"
            f"그룹: {etf_profile.get('group')} / 자산군: {etf_profile.get('asset_class')} / 카테고리: {etf_profile.get('category')}\n"
            f"투자 포커스: {etf_profile.get('focus')}\n"
            f"역할: {etf_profile.get('role_in_project')}\n"
            f"거시 민감도: {etf_profile.get('macro_sensitivity')}\n"
            f"위험 프로필: {etf_profile.get('risk_profile')}\n"
        )

    # 거시 텍스트 요약
    macro_text = ""
    if macro_context:
        if isinstance(macro_context, dict):
            macro_lines = [f"- {k}: {v}" for k, v in macro_context.items()]
            macro_text = "\n".join(macro_lines)
        else:
            macro_text = str(macro_context)

    # 점수 텍스트
    if consistency_score is None:
        consistency_score_text = "점수 없음"
    else:
        try:
            consistency_score_text = f"{float(consistency_score):.2f}"
        except Exception:
            consistency_score_text = str(consistency_score)

    prompt = f"""
    당신은 증권사 리서치 센터의 ETF 담당 애널리스트입니다.
    아래 정보를 종합해, 가격 비교 결과를 설명하세요. 인과관계가 검증되었다고 쓰지 마세요.
    가격 컨텍스트에 계산 불가 또는 관측 대기라고 표시된 ETF의 수익률과 영향은 추정하지 마세요.
    관측 기준시각: {state.get("as_of", "미지정")}
    가격창 상태: {state.get("price_window_status", {})}

    [사용자 질문]
    {question}

    [ETF 프로필]
    {etf_profile_text or "ETF 프로필 정보 없음"}

    [ETF 통계/지표 요약]
    {etf_stats_text}

    [가격 요약 정보]
    {price_ctx or "가격 요약 정보 없음"}

    [거시 환경 요약(있다면)]
    {macro_text or "거시 환경 요약 없음"}

    [뉴스 영향 분석(중간 결과)]
    {impact or "뉴스 영향 분석 정보 없음"}

    [뉴스-가격 일관성 평가]
    - 정량 점수: {consistency_score_text}
    - 코멘트: {consistency_comment or "설명 없음"}

    아래 형식을 가급적 따라 작성해 주세요:

    1. **이벤트 및 시장 환경 요약**
    - 이번 분석의 핵심이 되는 거시 이벤트(예: 금리 인하, CPI 발표)를 2~3줄로 정리.
    - 이벤트 발생일과 방향(서프라이즈 여부 포함)을 간단히 언급.

    2. **ETF 가격 반응 정량 분석**
    - 분석 대상 ETF의 최근 가격 흐름을 요약하고,
    - 이벤트 윈도우(예: 며칠 전~며칠 후) 수익률이 어느 정도인지 정량적으로 설명.
    - 필요하면 과거 유사 이벤트 대비 이번 반응이 '강한 편/평균적/약한 편'인지 언급.

    3. **뉴스 → 가격 전이 메커니즘 설명**
    - ETF 구조(섹터, 자산군, 테마)와 거시 민감도를 바탕으로,
    - 어떤 뉴스/이벤트가 어떤 채널(금리, 유동성, 위험선호, 실물 수요 등)을 통해
        가격에 영향을 주었는지 논리적으로 풀어 설명.
    - GLD vs QQQ vs XLF처럼 자산군이 다를 경우, 각 자산별로 메커니즘 차이를 짚어주세요.

    4. **뉴스-가격 일관성 평가**
    - 일관성 점수와 코멘트를 바탕으로,
        - (1) 이번 가격 움직임이 뉴스/이벤트로 어느 정도까지 설명 가능한지,
        - (2) 설명되지 않는 잔여 부분에 어떤 요인(테마 수급, 기술적 요인, 기타 이벤트 등)이 있을 수 있는지 논리적으로 제안.
    - "잘 설명됨 / 부분적으로 설명됨 / 설명력이 낮음" 정도의 등급을 붙여 요약.

    5. **투자 관점 인사이트 및 체크 포인트**
    - 단기(수일~수주)와 중기(분기) 관점에서 각각,
        - 현재 가격 위치가 과도/보수적 어느 쪽에 가까운지,
        - 추가로 확인해야 할 핵심 변수 2~3개(예: 향후 FOMC, EPS, 지정학 리스크)를 제시.
    - 마지막에 2~3줄로 한 문단 요약:
        - "요약: 이번 이벤트에 대한 [ETF명]의 가격 반응은 과거 대비 [강도] 수준이며,
            향후 [핵심 변수1, 변수2]에 따라 추가 리프라이싱 가능성이 존재함" 같은 형태로 정리.
    6. 반드시 아래 거시 지표들의 변화가 ETF 가격에 어떤 영향을 주었는지 서술하세요:
    - S&P500 방향성 (시장 위험선호)
    - VIX 변화 (공포/안정)
    - 10Y 금리 변화 (할인율)
    - DXY 변화 (달러 강세/약세)
    - GOLD 변화 (안전자산 선호)
    - BTC 변화 (디지털 자산 시장 심리)


    전문 애널리스트 리포트처럼, 과장된 표현을 피하고 
    근거 기반의 차분한 톤으로 **한국어**로 작성해 주세요.
    """

    answer = llm.invoke(prompt).content
    state["final_answer"] = answer
    return state

def stringify_macro_context(macro):
    if not macro:
        return "거시 지표 없음"

    spx = macro.get("SPX", {})
    vix = macro.get("VIX", {})
    ust = macro.get("UST10Y", {})
    gold = macro.get("GOLD", {})
    btc = macro.get("BTC", {})
    
    return f"""
    - S&P500: {spx.get('start')} → {spx.get('end')} 
      ({spx.get('chg_pct'):+.2f}%)
    - VIX(변동성): {vix.get('start')} → {vix.get('end')}
      ({vix.get('chg_pct'):+.2f}%)
    - 10Y 금리: {ust.get('start')}bp → {ust.get('end')}bp
      (변화: {ust.get('chg_bp'):+.2f}bp)
    - GOLD: {gold.get('start')} → {gold.get('end')}
      ({gold.get('chg_pct'):+.2f}%)
    - BTC: {btc.get('start')} → {btc.get('end')}
      ({btc.get('chg_pct'):+.2f}%)
    """



def attach_price_context(state: GraphState) -> GraphState:
    # Invalid or not-yet-published news must not enter the later explanation.
    cutoff = market_timestamp(state["as_of"])
    observed, published_dates = [], []
    for hit in state.get("reranked_hits", []):
        try:
            published = market_timestamp(hit.get("date"))
        except (ValueError, TypeError):
            continue
        if published <= cutoff:
            observed.append(hit)
            published_dates.append(published)
    empty = {"reranked_hits": observed, "price_context": "분석 기준시점 이전의 유효한 기사 없음.",
             "matched_etfs": [], "etf_stats": {}, "price_window_status": {}}
    if not observed:
        return empty
    text_all = " ".join((d.get("title", "") + " " + d.get("text", "")) for d in observed)
    matched = extract_etfs_from_text(text_all)
    if not matched:
        return {**empty, "price_context": "관련 ETF 매핑 없음."}
    reference = max(published_dates)
    horizon = state.get("horizon_sessions", 3)
    summaries, stats, statuses = [], {}, {}
    for ticker in matched:
        current = get_price_window(ticker, reference, window=horizon, as_of=cutoff)
        statuses[ticker] = current
        if current["status"] != "complete":
            summaries.append(f"{ticker}: 기사 이후 가격창 계산 불가/관측 대기 "
                             f"({current['status']}); 기사 {reference.isoformat()}, 관측 기준 {cutoff.isoformat()}")
            continue
        hist_fmt, hist_changes = summarize_historical_etf_reactions(
            ticker, text_all, reference, as_of=cutoff, horizon_sessions=horizon)
        hist_text = "; ".join(hist_fmt) if hist_fmt else "완료된 과거 비교 구간 없음"
        summaries.append(f"{ticker}: {current['start_date']}~{current['end_date']} "
                         f"{current['pct_change']:+.2f}% ({current['start_price']:.2f}→{current['end_price']:.2f}) / "
                         f"기사 이후 {horizon}개 관측 종가 / 과거 유사 기사 가격변화: {hist_text}")
        stats[ticker] = {"current_change": current["pct_change"],
                         "current_window": (current["start_date"], current["end_date"]),
                         "event_at": current["event_at"], "as_of": current["as_of"],
                         "hist_changes": hist_changes, "hist_formatted": hist_fmt}
    return {"reranked_hits": observed, "price_context": "\n".join(summaries),
            "matched_etfs": matched, "etf_stats": stats, "price_window_status": statuses}


def extract_etfs_from_text(text: str):
    if not text:
        return []
    found = set()
    low = text.lower()

    # 1) 직접 티커 문자열 매칭 (대소문자 무시)
    for t in KNOWN_TICKERS:
        if re.search(rf"\b{re.escape(t.lower())}\b", low):
            found.add(t)

    # 2) 한글/영문 키워드 → 티커 매핑
    for pat, tickers in KEYWORD_TO_TICKERS.items():
        if re.search(pat, text, flags=re.IGNORECASE):
            for t in tickers:
                if t in KNOWN_TICKERS:
                    found.add(t)

    return sorted(found)


def find_similar_news(current_text: str, ref_date, top_k=HIST_TOPK,
                      min_days_gap=HIST_MIN_DAYS_GAP, *, as_of):
    if not current_text:
        return []
    reference = market_timestamp(ref_date)
    cutoff = market_timestamp(as_of)
    candidates = vectorstore.similarity_search(current_text, k=max(top_k * 5, top_k))
    out, seen = [], set()
    for document in candidates:
        raw = document.metadata.get("date")
        try:
            published = market_timestamp(raw)
        except (ValueError, TypeError):
            continue
        if published > cutoff or published > reference - pd.Timedelta(days=min_days_gap):
            continue
        key = document.metadata.get("url") or document.page_content
        if key in seen:
            continue
        seen.add(key)
        out.append(document)
        if len(out) == top_k:
            break
    return out


def summarize_historical_etf_reactions(etf: str, current_text: str, ref_date,
                                        top_k=5, min_days_gap=7, *, as_of,
                                        horizon_sessions=3):
    documents = find_similar_news(current_text, ref_date, top_k=top_k,
                                  min_days_gap=min_days_gap, as_of=as_of)
    rows, seen = [], set()
    for document in documents:
        info = get_price_window(etf, document.metadata["date"], window=horizon_sessions, as_of=as_of)
        if info["status"] != "complete":
            continue
        key = (info["start_date"], info["end_date"])
        if key in seen:
            continue
        seen.add(key)
        rows.append(info)
    rows = sorted(rows, key=lambda r: r["end_date"], reverse=True)[:3]
    formatted = [f"{r['start_date']}~{r['end_date']} {r['pct_change']:+.2f}% "
                 f"({r['start_price']:.2f}→{r['end_price']:.2f})" for r in rows]
    return formatted, [r["pct_change"] for r in rows]

def attach_historical_price_context(state: dict) -> dict:
    """
    과거 유사 기사 기반의 보조 블록/통계만 추가한다.
    - 기존 price_context는 절대 빈 값으로 덮어쓰지 않는다.
    - etf_stats가 비면 아무것도 계산하지 않고 패스스루.
    """
    price_ctx = state.get("price_context", "")
    stats = state.get("etf_stats", {})

    # etf_stats가 없으면 그대로 통과
    if not stats:
        return {
            "historical_block": "",
            "etf_stats": stats,
            "price_context": price_ctx,   # 패스스루
        }

    # (예시) 간단 보조 요약 만들기 — 네가 쓰던 포맷 유지 가능
    lines = []
    for tkr, v in stats.items():
        curr = float(v.get("current_change", 0.0))
        hist = v.get("hist_changes", [])
        # 방향 일치율(예시)
        if hist:
            same_dir = sum(1 for h in hist if (h == 0 and curr == 0) or (h > 0 and curr > 0) or (h < 0 and curr < 0))
            score = round(same_dir / len(hist), 2)
            lines.append(f"- {tkr}: 과거 반응 일치율 {score} (표본 {len(hist)}건)")
        else:
            lines.append(f"- {tkr}: 과거 표본 없음")

    hist_block = "🕰️ 과거 유사 기사 반응 요약\n" + "\n".join(lines)

    # ⚠️ 여기서 절대 price_context를 빈 값으로 덮지 않도록 패스스루 적용
    writes = {
        "historical_block": hist_block,
        "etf_stats": stats,
        "price_context": price_ctx,   # 그대로 전달 (추가로 합치고 싶으면 아래처럼 결합)
    }
    return _passthrough(state, writes, keys=("price_context","etf_stats"))

def score_consistency(state: dict) -> dict:
    """
    뉴스와 가격 반응의 일관성을 계산하는 노드.
    state:
      - reranked_hits: 상위 뉴스들
      - price_context / historical price context 등
    """

    hits = state.get("reranked_hits", [])
    price_ctx = state.get("price_context", "")

    # --- (예시) 아주 단순한 점수 계산 ---
    # 실제 프로젝트에서는 너가 필요하면 더 지능적으로 바꿔도 됨.

    if not hits or not price_ctx:
        state["consistency_score"] = None
        state["consistency_comment"] = "일관성을 계산할 데이터가 부족합니다."
        return state

    # 예: 텍스트 기반 간단 매칭으로 score 구성 (더 개선 가능)
    text_blob = " ".join([h.get("text", "") for h in hits])
    score = 0.0

    # 키워드 기반 점수 예시
    if "상승" in price_ctx:
        if any(k in text_blob for k in ["호재", "금리 인하", "성장", "매출 증가"]):
            score += 0.7
        if "긴축" in text_blob:
            score -= 0.4

    elif "하락" in price_ctx:
        if any(k in text_blob for k in ["리스크", "전쟁", "인플레이션", "우크라이나", "중동"]):
            score += 0.7
        if "호재" in text_blob:
            score -= 0.4

    final_score = max(0.0, min(1.0, score))  # 0~1로 클램핑

    # 설명
    if final_score > 0.6:
        comment = "뉴스 내용이 가격 움직임과 비교적 잘 들어맞습니다."
    elif final_score > 0.3:
        comment = "일부는 부합하지만 가격 움직임을 완전히 설명하기에는 부족합니다."
    else:
        comment = "뉴스 내용과 가격 흐름의 직접적 연결성이 낮습니다."

    # 👉 이 두 줄이 핵심!
    state["consistency_score"] = final_score
    state["consistency_comment"] = comment

    return state



def fuse_results(state) -> Dict[str, Any]:
    """
    bm25_hits / dense_hits / web_hits(옵션) 를 하나로 합쳐 랭크.
    - 키 중복(같은 url/title)은 최고 점수로 통합
    - 가중치 기본: dense 0.5, bm25 0.35, web 0.15
    """
    bm25 = state.get("bm25_hits") or []
    dense = state.get("dense_hits") or []
    web   = state.get("web_hits") or []   # retrieve_web이 만든 리스트(있을 때만)

    W = {"dense": 0.50, "bm25": 0.35, "web": 0.15}

    bucket: Dict[str, Dict[str, Any]] = {}

    def _key(h):
        return h.get("url") or (h.get("title") or "")[:80]

    def _accumulate(h):
        k = _key(h)
        if not k:
            return
        if k not in bucket:
            bucket[k] = {**h, "score_bm25":0.0, "score_dense":0.0, "score_web":0.0}
        origin = h.get("origin")
        if origin == "bm25":
            bucket[k]["score_bm25"] = max(bucket[k]["score_bm25"], float(h["score"]))
        elif origin == "dense":
            bucket[k]["score_dense"] = max(bucket[k]["score_dense"], float(h["score"]))
        elif origin == "web":
            bucket[k]["score_web"] = max(bucket[k]["score_web"], float(h["score"]))
        # 메타 최신화(비어있으면 채움)
        for fld in ("title","text","date","source","url"):
            if not bucket[k].get(fld) and h.get(fld):
                bucket[k][fld] = h.get(fld)

    for h in bm25: _accumulate(h)
    for h in dense: _accumulate(h)
    for h in web:   _accumulate(h)

    fused = []
    for k, h in bucket.items():
        fused_score = (W["dense"]*h["score_dense"] +
                       W["bm25"] *h["score_bm25"]  +
                       W["web"]  *h["score_web"])
        h_out = {**h, "score": round(float(fused_score), 4), "origin":"fused"}
        fused.append(h_out)

    # 점수 내림차순
    fused.sort(key=lambda x: x["score"], reverse=True)

    # 상위 N개만 전달 (후속 rerank가 있으므로 30개 정도)
    TOP = 30
    return {"fused_hits": fused[:TOP]}

def _doc_to_hit(d: Document, score: float, origin: str) -> Dict[str, Any]:
    return {
        "id": d.metadata.get("url") or d.metadata.get("title") or id(d),
        "title": d.metadata.get("title") or d.page_content[:60],
        "text": d.page_content,                     # ← 반드시 채움
        "date": d.metadata.get("date"),
        "source": d.metadata.get("source") or origin,
        "url": d.metadata.get("url"),
        "score": float(score),
        "origin": origin,
    }

### 실험용으로 유지
def generate_answer(state: GraphState) -> GraphState:
    etf_stats: Dict[str, Any] = state.get("etf_stats", {}) or {}
    price_ctx: str = state.get("price_context", "") or ""
    reranked = state.get("reranked_hits", []) or []
    consistency_block = state.get("consistency_block", "") or ""

    # 1) 핵심 ETF 요약 (현재 변동률 기준 정렬)
    rows = []
    for etf, st in etf_stats.items():
        rows.append({
            "etf": etf,
            "cur": float(st.get("current_change", 0.0)),
            "cur_win": st.get("current_window", ("?", "?")),
            "hist_avg": (
                sum(st.get("hist_changes", []) or []) / max(1, len(st.get("hist_changes", [])))
            ) if st.get("hist_changes") else None,
        })

    # 🔸 영향도 큰 순서로 정렬
    rows = sorted(rows, key=lambda x: abs(x["cur"]), reverse=True)

    # 🔸 % 포맷팅 함수
    def _pct(v): 
        return "N/A" if v is None else f"{v:+.2f}"

    # 표 형식 출력
    lines = []
    if rows:
        lines.append("📊 영향 추정 상위 ETF")
        lines.append("ETF | 현재 5영업일 변동(%) | 윈도우 | 과거 유사 반응 평균(%)")
        lines.append("---|---:|---|---:")
        for r in rows[:8]:
            cur = _pct(r["cur"])
            win = f"{r['cur_win'][0]}~{r['cur_win'][1]}"
            havg = _pct(r["hist_avg"])
            lines.append(f"{r['etf']} | {cur} | {win} | {havg}")
        lines.append("")

    # 2) 가격 컨텍스트(티커별 서술) 붙이기
    if price_ctx:
        lines.append("🧾 가격 컨텍스트")
        lines.append(price_ctx.strip())
        lines.append("")

    # 3) 참고 기사(상위 5개)
    if reranked:
        lines.append("📰 참고 기사(상위)")
        for i, d in enumerate(reranked[:5], start=1):
            title = (d.get("title") or "").strip()
            url = d.get("url") or ""
            src = d.get("source") or d.get("origin") or "doc"
            date = d.get("date") or ""
            lines.append(f"{i}. [{title}]({url}) · {src} · {date}")
        lines.append("")

    # 4) 일관성 블록(있으면)
    if consistency_block:
        lines.append("✅ 히스토리컬 일관성 요약")
        lines.append(consistency_block.strip())
        lines.append("")

    final = "\n".join(lines).strip() or "분석 가능한 컨텍스트가 충분하지 않습니다."
    return {"final_answer": final}

def attach_macro_context(state: dict) -> dict:
    global macro_df
    if macro_df is None or macro_df.empty:
        state["macro_context"] = {}
        return state

    reactions = state.get("price_reactions", []) or []
    if not reactions:
        state["macro_context"] = {}
        return state
    window_days = state.get("price_window_days", 3)
    today_str = state.get("today", None)

    # 날짜 범위 잡기
    from datetime import datetime, timedelta

    def _parse_date(s: str):
        if isinstance(s, str):
            for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y-%m-%dT%H:%M:%S"):
                try:
                    return datetime.strptime(s, fmt).date()
                except ValueError:
                    continue
        return None

    start_date = end_date = None
    if reactions:
        r = reactions[-1]
        s = getattr(r, "start_date", None) or (r.get("start_date") if isinstance(r, dict) else None)
        e = getattr(r, "end_date", None) or (r.get("end_date") if isinstance(r, dict) else None)
        start_date = _parse_date(s)
        end_date = _parse_date(e)

    if start_date is None or end_date is None:
        if today_str:
            today = _parse_date(today_str)
        else:
            today = macro_df["date"].max().date()
        end_date = today
        start_date = today - timedelta(days=window_days)

    # 날짜 필터
    df = macro_df.copy()
    df["date_only"] = df["date"].dt.date
    mask = (df["date_only"] >= start_date) & (df["date_only"] <= end_date)
    win = df[mask].sort_values("date_only")

    if win.empty:
        state["macro_context"] = {}
        return state

    import pandas as pd

    def _range_change(col: str):
        """해당 컬럼의 기간 내 시작/끝/변화량/변화율 계산 (스칼라만 사용하도록 방어적으로 처리)."""
        if col not in win.columns:
            return None, None, None, None

        s = win[col]
        # 만약 DataFrame이나 Series 벡터면 첫 번째 컬럼만 사용
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]

        s = s.dropna()
        if s.empty:
            return None, None, None, None

        start_val = s.iloc[0]
        end_val = s.iloc[-1]

        # 숫자로 변환 가능한 경우만
        try:
            start_val_f = float(start_val)
            end_val_f = float(end_val)
        except Exception:
            return None, None, None, None

        diff = end_val_f - start_val_f
        pct = (end_val_f / start_val_f - 1.0) * 100 if start_val_f != 0 else None
        return start_val_f, end_val_f, diff, pct

    # 여기서부터는 그대로
    spx_start, spx_end, spx_diff, spx_pct = _range_change("SPX_close")
    vix_start, vix_end, vix_diff, vix_pct = _range_change("VIX_close")
    dxy_start, dxy_end, dxy_diff, dxy_pct = _range_change("DXY_close")
    ust_start, ust_end, ust_diff, _ = _range_change("UST10Y_close")
    gold_start, gold_end, gold_diff, gold_pct = _range_change("GOLD_close")
    btc_start, btc_end, btc_diff, btc_pct = _range_change("BTC_close")

    macro_summary = {
        "window_start": start_date.isoformat(),
        "window_end": end_date.isoformat(),
        "SPX": {
            "start": spx_start,
            "end": spx_end,
            "chg_abs": spx_diff,
            "chg_pct": spx_pct,
        },
        "VIX": {
            "start": vix_start,
            "end": vix_end,
            "chg_abs": vix_diff,
            "chg_pct": vix_pct,
        },
        "DXY": {
            "start": dxy_start,
            "end": dxy_end,
            "chg_abs": dxy_diff,
            "chg_pct": dxy_pct,
        },
        "UST10Y": {
            "start": ust_start,
            "end": ust_end,
            "chg_bp": ust_diff,
        },
        "GOLD": {
            "start": gold_start,
            "end": gold_end,
            "chg_abs": gold_diff,
            "chg_pct": gold_pct,
        },
        "BTC": {
            "start": btc_start,
            "end": btc_end,
            "chg_abs": btc_diff,
            "chg_pct": btc_pct,
        },
    }

    state["macro_context"] = macro_summary
    return state


옵션) 조건 반복 노드

In [ ]:
def route_has_candidates(state: GraphState) -> str:
    total = len(state.get("bm25_hits", [])) + len(state.get("dense_hits", [])) + len(state.get("web_hits", []))
    return "ok" if total > 0 else "no_data"

## 그래프 빌드

In [ ]:
builder = StateGraph(GraphState)
memory = MemorySaver()
# 노드 추가
builder.add_node("init_today", init_today)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("retrieve_bm25", retrieve_bm25)
builder.add_node("retrieve_dense", retrieve_dense)
builder.add_node("retrieve_web", retrieve_web)
builder.add_node("fuse_results", fuse_results)
builder.add_node("rerank_results", rerank_results)
builder.add_node("attach_price_context", attach_price_context)
builder.add_node("attach_historical_price_context", attach_historical_price_context)
builder.add_node("score_consistency", score_consistency)
builder.add_node("analyze_price_reaction", analyze_price_reaction)
builder.add_node("explain_news_impact", explain_news_impact)
builder.add_node("final_answer", final_answer)
builder.add_node("attach_etf_profile", attach_etf_profile)
builder.add_node("attach_macro_context", attach_macro_context)

# 실험용
builder.add_node("generate_answer", generate_answer)

# 엣지
builder.add_edge(START, "init_today")
builder.add_edge("init_today", "rewrite_query")
builder.add_edge("rewrite_query", "retrieve_bm25")
builder.add_edge("rewrite_query", "retrieve_dense")
builder.add_edge("rewrite_query", "retrieve_web")

builder.add_conditional_edges(
    "retrieve_web",
    route_has_candidates,
    {"ok": "fuse_results", "no_data": "fuse_results"}
)
builder.add_edge("retrieve_bm25", "fuse_results")
builder.add_edge("retrieve_dense", "fuse_results")

# 1) 리랭킹 (bge-m3)
builder.add_edge("fuse_results", "rerank_results")

# 2) 가격 컨텍스트 붙이기 (기존 그대로 활용)
builder.add_edge("rerank_results", "attach_price_context")
builder.add_edge("attach_price_context", "attach_etf_profile")
builder.add_edge("attach_etf_profile", "attach_historical_price_context")

# 3) 일관성 점수도 계속 쓰고 싶으면 여기서 계산 + 거시 데이터
builder.add_edge("attach_historical_price_context", "analyze_price_reaction")
builder.add_edge("analyze_price_reaction", "attach_macro_context")
builder.add_edge("attach_macro_context", "score_consistency")

# 4) 가격 반응 분석 (새 노드)
builder.add_edge("score_consistency", "explain_news_impact")

# 5) 뉴스 영향 설명 (새 노드)
# Price reactions are attached before macro/consistency analysis.

# 6) 최종 인사이트 생성 (generate_answer 대신 final_answer)
builder.add_edge("explain_news_impact", "final_answer")
builder.add_edge("final_answer", END)


컴파일

In [ ]:
graph = builder.compile(checkpointer=memory)

print(graph.get_graph().draw_mermaid())

## 테스트 코드

### 성능 점검

리트리버별 점검

In [ ]:
cfg = RunnableConfig(configurable={"thread_id": CFG.thread_id}, recursion_limit=20)

question = "최근 금리 인하 뉴스가 ETF에 어떤 영향을 줬는지 알려줘"

state = graph.invoke({"as_of": datetime.now(timezone.utc).isoformat(), "question": question}, config=cfg)

print("state keys:", state.keys())
print("\n[final_answer]\n", state.get("final_answer", "")[:800])

print("\n[bm25_hits 개수]", len(state.get("bm25_hits", [])))
print("[dense_hits 개수]", len(state.get("dense_hits", [])))
print("[web_hits 개수]", len(state.get("web_hits", [])))

어떤 문서가 사용되었는가

In [ ]:
for i, hit in enumerate(state.get("reranked_hits", [])[:5], start=1):
    print(f"\n=== Reranked Hit {i} | score={hit.get('score'):.4f} | source={hit.get('source')} | origin={hit.get('origin')} ===")

    doc = hit.get("doc")

    if doc is not None:
        # LOCAL_DOCS 기반 (BM25/FAISS에서 온 것)
        title = doc.metadata.get("title")
        date  = doc.metadata.get("date")
        url   = doc.metadata.get("url")
        text  = doc.page_content
    else:
        # NAVER/Web 기반 hit
        title = hit.get("title") or "(제목 없음)"
        date  = hit.get("date")  or ""
        url   = hit.get("url")   or ""
        text  = hit.get("text")  or ""

    print("제목:", title)
    print("날짜:", date)
    print("URL:", url)
    print("본문 일부:", str(text)[:200].replace("\n", " "))


앙상블 리트리버 점검

In [ ]:
print("price_df shape:", price_df.shape)
print("cols:", price_df.columns.tolist())
print("dtypes:", price_df.dtypes.to_dict())

# 날짜 변환(강제), 정규화(대문자/트림)
price_df["date_kst"] = pd.to_datetime(price_df["date_kst"], errors="coerce")
price_df["ticker"] = price_df["ticker"].astype(str).str.strip().str.upper()

print("date_kst nulls:", int(price_df["date_kst"].isna().sum()))
print("tickers:", sorted(price_df["ticker"].unique().tolist()))
print("date range:", price_df["date_kst"].min(), "→", price_df["date_kst"].max())

print('--------------------------')
ref = "2025-11-01"  # 뉴스 샘플 날짜 근처
for t in ["BITO", "GLD", "SLV", "XLE", "QQQ", "SPY", "XLF"]:
    print(t, "→", get_price_window(t, ref, window=5))

print('--------------------------')
state = graph.invoke({"as_of": datetime.now(timezone.utc).isoformat(), "question":"오늘 금리 인하 뉴스 영향 ETF 알려줘"}, config=cfg)
print("price_ctx exists:", bool(state.get("price_context")))
print(state.get("price_context","")[:400])
print("etf_stats keys:",list(state.get("etf_stats",{}).keys()))

bge-m3 컴퓨터 스코어 작동 테스트

In [ ]:
score = model.compute_score([["금리 인하가 ETF 가격에 미치는 영향", 
                              "미국 기준금리가 인하되며 기술주 ETF가 상승했다."]])

print(score)

### 디버깅 테스트

ETF_STATS 점검

In [ ]:
etf_stats = state.get("etf_stats", {})
print("etf_stats keys (tickers):", list(etf_stats.keys()))

for t, stats in list(etf_stats.items())[:3]:  # 앞에서 3개만
    print(f"\n[{t}] raw stats:")
    print(json.dumps(stats, indent=2, ensure_ascii=False, default=str))

plt 그래프 테스트

In [ ]:
cfg = RunnableConfig(configurable={"thread_id": CFG.thread_id}, recursion_limit=30)
state = graph.invoke({"as_of": datetime.now(timezone.utc).isoformat(), "question": "오늘 금리 인하 뉴스 영향 ETF 알려줘"}, config=cfg)

fig_paths = save_visuals(
    state,
    price_df,
    out_dir="reports/figs",
    recent_days=60,
    event_pre_days=1,   # 이벤트 하루 전부터
    event_post_days=3,  # 이벤트 이후 3일
    make_event_overlay=True,
    make_event_hybrid=True,
)

print("총 이미지 수:", len(fig_paths))
print([p for p in fig_paths if "event" in p][:10])

price_ctx exists 문제 테스트

In [ ]:
print("price_df shape:", price_df.shape)
print("cols:", price_df.columns.tolist())
print("dtypes:", price_df.dtypes.to_dict())

# 날짜 변환(강제), 티커 정규화(대문자/트림)
price_df["date_kst"] = pd.to_datetime(price_df["date_kst"], errors="coerce")
price_df["ticker"] = price_df["ticker"].astype(str).str.strip().str.upper()

print("date_kst nulls:", int(price_df["date_kst"].isna().sum()))
print("tickers:", sorted(price_df["ticker"].unique().tolist()))
print("date range:", price_df["date_kst"].min(), "→", price_df["date_kst"].max())


In [ ]:
ref = "2025-11-01"
for t in ["BITO","GLD","SLV","XLE","QQQ","SPY","XLF"]:
    print(t, "→", get_price_window(t, ref, window=5))

네이버 크롤링 확인

In [ ]:
state = graph.invoke({"as_of": datetime.now(timezone.utc).isoformat(), "question":"오늘 ETF에 영향을 주는 뉴스 영향 알려줘"}, config=cfg)
print("len fused:", len(state.get("fused_hits", [])))
print("len reranked:", len(state.get("reranked_hits", [])))
print("price_ctx exists:", bool(state.get("price_context")))
print("price_context preview:\n", state.get("price_context","")[:400])
print("etf_stats keys:", list(state.get("etf_stats", {}).keys()))

노드별 스테이트 변화

In [ ]:
cfg = RunnableConfig(configurable={"thread_id": "debug-stream"}, recursion_limit=20)

question = "오늘 금리 관련 이슈가 ETF 가격에 어떤 영향을 줬는지 알려줘"

print("=== 그래프 스트림 디버깅 ===")
for event in graph.stream({"as_of": datetime.now(timezone.utc).isoformat(), "question": question}, config=cfg):
    print("\n------------------------")
    print(event)

거시 데이터 삽입 노드 테스트

In [ ]:
test_state = {
    "price_reactions": state.get("price_reactions", []),  # 기존 실행 state에서 가져오거나
    "today": state.get("today", None),
    "price_window_days": 3
}
attach_macro_context(test_state)
test_state["macro_context"]

## 엔드 투 엔드

In [ ]:
cfg = RunnableConfig(configurable={"thread_id": CFG.thread_id}, recursion_limit=30)
state = graph.invoke({"as_of": datetime.now(timezone.utc).isoformat(), "question": "오늘 경제 뉴스와 영향받을만한 ETF 알려줘"}, config=cfg)

fig_paths = save_visuals(state, price_df, out_dir="reports/figs")

report_path = save_report_md(state, out_dir="reports", prefix="etf_report", fig_paths=fig_paths)

# 🔹 로그 저장
log_paths = save_run_logs(state, out_dir="logs", prefix="run")

print("Report saved:", report_path)
print("Figures:", fig_paths)
print("Logs saved:", log_paths)

발표 자료용 코드

In [ ]:
def show_meta(list_name):
    docs = state.get(list_name, [])
    print(f"\n=== {list_name} ({len(docs)} docs) ===\n")
    for i, doc in enumerate(docs[:5]):
        if isinstance(doc, dict):
            meta = doc.get("metadata", None)
            if meta:
                print(f"[{i}] metadata:")
                for k, v in meta.items():
                    print(f"  {k}: {v}")
            else:
                print(f"[{i}] dict:", doc)
        else:
            # Document 타입일 때
            print(f"[{i}] Document.metadata:", doc.metadata)
        print("-"*40)

show_meta("bm25_hits")
show_meta("dense_hits")
show_meta("reranked_hits")

In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd

# ETF 선택
ticker = "QQQ"

# 뉴스 이벤트 날짜
event_date = "2024-11-13"
event_label = "CPI 발표"

# 데이터 범위
start = "2024-10-25"
end   = "2024-11-25"

df = yf.download(ticker, start=start, end=end)

plt.figure(figsize=(12,6))
plt.plot(df.index, df["Close"], linewidth=2)

# 이벤트 표시
plt.axvline(pd.to_datetime(event_date), color="red", linestyle="--", linewidth=2)
plt.text(pd.to_datetime(event_date), df["Close"].max()*0.98, 
         event_label, rotation=90, color="red", fontsize=12)

plt.title(f"{ticker} 가격과 뉴스 이벤트", fontsize=16)
plt.xlabel("날짜")
plt.ylabel("종가")
plt.grid(True)
plt.tight_layout()
plt.show()
